In [18]:
!pip install pyspark

In [19]:
# Kell Drinkwater
# CS131 Su26
# ws5

#A1. Create a SparkSession named ws5-regression.

import sys
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ws5-regression").getOrCreate()

In [20]:
#A2. Read the dataset from your bucket into a DataFrame with the header row
#as column names and column types inferred, then .show() it.
#Don't hardcode the bucket — take the gs://.../tips.csv path as a command-line
#argument (sys.argv[1]).
#path = sys.argv[1]

# this data comes from
# https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv

path = "/content/tips.csv"
inputDF = spark.read.csv(path, header=True, inferSchema=True)

inputDF.printSchema()
print(inputDF.count(), "total records")
inputDF.show(10)

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)

244 total records
+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
+----------+----+------+------+---+------+----+
only showing top 10 rows


In [21]:
#A3. Combine the two predictor columns total_bill and size into a single
#vector column called features.

from pyspark.ml.feature import VectorAssembler
va = VectorAssembler(inputCols=["total_bill", "size"], outputCol="features")
vecDF = va.transform(inputDF)
vecDF.select("total_bill", "size", "features", "tip").show(5)

+----------+----+-----------+----+
|total_bill|size|   features| tip|
+----------+----+-----------+----+
|     16.99|   2|[16.99,2.0]|1.01|
|     10.34|   3|[10.34,3.0]|1.66|
|     21.01|   3|[21.01,3.0]| 3.5|
|     23.68|   2|[23.68,2.0]|3.31|
|     24.59|   4|[24.59,4.0]|3.61|
+----------+----+-----------+----+
only showing top 5 rows


In [22]:
#A4. Split the data into 80% train / 20% test. Pass a fixed seed so the
#split is reproducible. (Hint: .randomSplit().)

#trainVecDF, testVecDF = vecDF.randomSplit([0.8, 0.2], seed=37)
#print(f"Training set {trainVecDF.count()} rows, test set {testVecDF.count()} rows")

trainDF, testDF = inputDF.randomSplit([0.8, 0.2], seed=37)

In [23]:
#A5. Define a LinearRegression with featuresCol="features" and labelCol="tip",
##and fit the model.

from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="tip")

#Chain the assembler (A3) and the regressor into a Pipeline
#and call .fit() on the training set.
pipeline = Pipeline(stages=[va, lr])
pipeModel = pipeline.fit(trainDF)

#A6. Apply the fitted pipeline to the test set to produce predictions.
predictDF = pipeModel.transform(testDF)

repredictTrainingDF = pipeModel.transform(trainDF)


In [24]:
#(Check predictions)
predictDF.printSchema()
predictDF.select("total_bill", "size", "features", "tip", "prediction").show(10)

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- prediction: double (nullable = false)

+----------+----+-----------+----+------------------+
|total_bill|size|   features| tip|        prediction|
+----------+----+-----------+----+------------------+
|      7.51|   2| [7.51,2.0]| 2.0|1.7652357795682923|
|      8.51|   2| [8.51,2.0]|1.25|1.8572736386768174|
|      9.68|   2| [9.68,2.0]|1.32|1.9649579338337917|
|      9.78|   2| [9.78,2.0]|1.73| 1.974161719744644|
|     10.59|   2|[10.59,2.0]|1.61| 2.048712385622549|
|     10.63|   2|[10.63,2.0]| 2.0|2.0523938999868903|
|     11.59|   2|[11.59,2.0]| 1.5|2.1407502447310742|
|     11.87|   2|[11.87,2.0]|1.63| 2.166520845281461|
|     12.02|   2|[12.02,2.0]|1.97|  2.18032652414774|
|    

In [25]:
#A7. Evaluate the predictions on two metrics: RMSE and R². Use one evaluator
#with the label column tip, changing metricName.

from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    predictionCol="prediction",
    labelCol="tip",
    metricName="rmse"
)
rmseResult = evaluator.evaluate(predictDF)
#r2Eval = RegressionEvaluator(
#    predictionCol="prediction",
#    labelCol="tip",
#    metricName="r2"
#)
#r2Result = r2Eval.evaluate(predictDF)
#r2Result = rmseEval.evaluate(predictDF, {metricName: "r2"})
evaluator.setMetricName("r2")
r2Result = evaluator.evaluate(predictDF)

In [26]:
#A8. Pull the fitted LinearRegression model out of the pipeline (pipelineModel.stages[-1])
#and print its coefficients and intercept, plus the RMSE and R² from A7.
#Use clear labels (e.g. print(f"RMSE: {rmse}")) so the numbers stand out in the job log.

fittedLR = pipeModel.stages[-1]
co1, co2 = fittedLR.coefficients
b = fittedLR.intercept

print(f"Coefficients: {co1:.2f}, {co2:.2f} / Intercept: {b:.2f}")
print(f"RMSE: {rmseResult:.2f}\nR^2: {r2Result:.2f}")


Coefficients: 0.09, 0.23 / Intercept: 0.61
RMSE: 0.99
R^2: 0.45


In [30]:
summary = fittedLR.summary
summary.residuals.show(5)
print(f"{summary.totalIterations} iterations")
print(f"{summary.objectiveHistory} objective history")
print(f"RMSE: {summary.rootMeanSquaredError:.2f}")
print(f"R^2: {summary.r2:.2f}")

+--------------------+
|           residuals|
+--------------------+
|-0.12458021643319617|
| -0.6032491475372885|
| -0.5092984675068304|
|  3.4086940637999246|
|-0.32983767252371865|
+--------------------+
only showing top 5 rows
0 iterations
[0.0] objective history
RMSE: 1.01
R^2: 0.47


In [28]:
#This cell's RMSE/R2 match the previous cell.
#I think that means the model.summary metrics are for how well the model
#fits the TRAINING data, while part A7 is checking how well the model fits
#the TEST data.

eval2 = RegressionEvaluator(
    predictionCol="prediction",
    labelCol="tip",
    metricName="rmse"
)

trainRMSE = eval2.evaluate(repredictTrainingDF)
eval2.setMetricName("r2")
trainR2 = eval2.evaluate(repredictTrainingDF)

print("Using evaluator on model on training DF")
print(f"RMSE: {trainRMSE:.2f}\nR^2: {trainR2:.2f}")

Using evaluator on model on training DF
RMSE: 1.01
R^2: 0.47
